# all-reduce-grad-sync — ex2: skip-sync optimization — only all_reduce non-None grads

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `all-reduce-grad-sync`. Running the final beacon cell reports progress against the `Distributed: all_reduce grad sync` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce grad sync` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-grad-sync`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-grad-sync"
DD_SUBTOPIC = "Distributed: all_reduce grad sync"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Skip-sync optimization — only all_reduce non-None grads

Ex1 looped every parameter and all_reduced its `.grad`. In real training, some parameters get NO gradient on this step:

- Frozen layers (`requires_grad=False`).
- Sparse models where this batch didn't touch certain heads.
- Embedding tables when none of this batch's tokens hit them.

Their `.grad` is `None`. Calling `dist.all_reduce(None, ...)` either crashes or silently sends a zero tensor — wasted bandwidth. The skip-sync optimization:

```python
for p in model.parameters():
    if p.grad is None:
        continue
    dist.all_reduce(p.grad, op=dist.ReduceOp.SUM)
    p.grad /= world_size
```

**Subtlety: rank consistency.** Every rank must agree on WHICH parameters to skip — if rank 0 skips `embedding.weight` but rank 1 does not, you deadlock (rank 1 hangs waiting for rank 0's all_reduce that never comes). In practice every rank runs the same code over the same model graph, so the `is None` check returns the same answer everywhere. If your grads diverge across ranks before sync, that's a bug ABOVE this layer.

**Why not iterate `model.named_parameters()` instead.** Same answer; `.parameters()` is the canonical form and you don't need the names for this op.

### Exercise 2 — skip-sync optimization — only all_reduce non-None grads

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the `if p.grad is None: continue` skip-sync optimization before `dist.all_reduce(p.grad, SUM)` + divide-by-world_size, verified by running on a model with one frozen layer.
> Keywords: DDP, grad-sync, skip-sync, frozen-layers, sparse-grads
> ```

**KCs targeted:** `skip-none-grads-in-sync-loop`, `all-reduce-mean-divide-after-skip`

Implement `ex2_grad_sync_skip_none(rank, world_size, dist_module, model)`. The frozen-layer-aware grad sync:

1. Loop `for p in model.parameters()`.
2. **If `p.grad is None`: `continue`.** This skips parameters that didn't receive a gradient on this step (frozen layers, sparse heads, etc.).
3. Otherwise, all-reduce the grad: `dist_module.all_reduce(p.grad, op=dist_module.ReduceOp.SUM)`.
4. Divide by `world_size` in-place: `p.grad /= world_size`.
5. Return the integer COUNT of parameters that were synced (i.e. had non-None grads). The test asserts this count matches expectations.

Important: every rank runs this same loop on the same model graph, so the `is None` decision is consistent across ranks — no risk of deadlock.

Input: `rank`, `world_size` — ints; `dist_module` — torch.distributed or mock; `model` — `nn.Module` with some parameters' `.grad` set, some left as `None`.
Output: `int` — count of parameters synced (i.e. that had a non-None grad).

In [ ]:
def ex2_grad_sync_skip_none(rank: int, world_size: int, dist_module, model: 'nn.Module') -> int:
    synced = 0
    for p in model.parameters():
        if p.grad is None:
            continue
        dist_module.all_reduce(p.grad, op=dist_module.ReduceOp.SUM)
        p.grad /= world_size
        synced += 1
    return synced


<details><summary>Solution</summary>

```python
def ex2_grad_sync_skip_none(rank: int, world_size: int, dist_module, model: 'nn.Module') -> int:
    synced = 0
    for p in model.parameters():
        if p.grad is None:
            continue
        dist_module.all_reduce(p.grad, op=dist_module.ReduceOp.SUM)
        p.grad /= world_size
        synced += 1
    return synced
```

**Why `is None`, not `== 0`.** A zero-VALUED grad tensor still needs to be reduced — it's just one rank's contribution that happens to be zero. `None` means 'no grad was computed this step' (e.g. backward was never called for this param's subgraph). Only the latter should be skipped.

**Rank consistency is load-bearing.** If rank 0 thinks `p.grad is None` and rank 1 thinks it has a grad, rank 1 calls `all_reduce` with no counterpart — deadlock. In practice every rank runs the same forward/backward on the same model graph, so the `is None` answer is identical. The invariant is preserved by the data-parallelism contract, not by any check in this function.

**Real DDP fuses + skips automatically.** `torch.nn.parallel.DistributedDataParallel` hooks backward and kicks an all_reduce when each param's grad becomes ready — skipping frozen layers naturally. The hand-rolled loop you're writing is the conceptual model; DDP is the optimized production version.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()